In [29]:
# Import necessary libraries and load the dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/StudentPerformanceFactors.csv")
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [30]:
# Identify numerical and categorical columns

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity', 'Exam_Score']
Categorical columns: ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [31]:
# Handle missing values

# numerical: fill with mean
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

# categorical: fill with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [32]:
df.isnull().sum()

Hours_Studied                 0
Attendance                    0
Parental_Involvement          0
Access_to_Resources           0
Extracurricular_Activities    0
Sleep_Hours                   0
Previous_Scores               0
Motivation_Level              0
Internet_Access               0
Tutoring_Sessions             0
Family_Income                 0
Teacher_Quality               0
School_Type                   0
Peer_Influence                0
Physical_Activity             0
Learning_Disabilities         0
Parental_Education_Level      0
Distance_from_Home            0
Gender                        0
Exam_Score                    0
dtype: int64

In [33]:
# Encode categorical variables using one-hot encoding

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_encoded.head()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score,Parental_Involvement_Low,Parental_Involvement_Medium,Access_to_Resources_Low,...,Teacher_Quality_Medium,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Parental_Education_Level_High School,Parental_Education_Level_Postgraduate,Distance_from_Home_Moderate,Distance_from_Home_Near,Gender_Male
0,23,84,7,73,0,3,67,True,False,False,...,True,True,False,True,False,True,False,False,True,True
1,19,64,8,59,2,4,61,True,False,False,...,True,True,False,False,False,False,False,True,False,False
2,24,98,7,91,2,4,74,False,True,False,...,True,True,True,False,False,False,True,False,True,True
3,29,89,8,98,1,4,71,True,False,False,...,True,True,False,False,False,True,False,True,False,True
4,19,92,6,65,3,4,70,False,True,False,...,False,True,True,False,False,False,False,False,True,False


In [34]:
# Separate features and target variable

X = df_encoded.drop('Exam_Score', axis=1)
y = df_encoded['Exam_Score']

In [35]:
# Scale numerical features

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [36]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((5285, 27), (1322, 27))

In [37]:
# Save the processed train and test datasets to CSV files

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv("../data/train_processed.csv", index=False)
test_data.to_csv("../data/test_processed.csv", index=False)

# Day 3 — Model Training

STEP 1 — Import ML libraries

In [38]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

STEP 2 — Load processed train/test data

In [39]:
train_data = pd.read_csv("../data/train_processed.csv")
test_data = pd.read_csv("../data/test_processed.csv")

X_train = train_data.drop("Exam_Score", axis=1)
y_train = train_data["Exam_Score"]

X_test = test_data.drop("Exam_Score", axis=1)
y_test = test_data["Exam_Score"]

# Ensure all values are numeric (prevents TypeError in evaluation)
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')
y_test = pd.to_numeric(y_test, errors='coerce')

STEP 3 — Create evaluation function

In [40]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5   # FIXED
    r2 = r2_score(y_test, y_pred)
    return mae, rmse, r2

STEP 4 — Train Linear Regression

In [41]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_mae, lr_rmse, lr_r2 = evaluate_model(lr, X_test, y_test)

print("Linear Regression")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R2:", lr_r2)

Linear Regression
MAE: 0.45239200896259785
RMSE: 1.8044445092722838
R2: 0.7696495724907313


STEP 5 — Train Random Forest

In [42]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

rf_mae, rf_rmse, rf_r2 = evaluate_model(rf, X_test, y_test)

print("Random Forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest
MAE: 1.164160363086233
RMSE: 2.214647528291388
R2: 0.6530146071824954


STEP 6 — Train Gradient Boosting

In [43]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)

gb_mae, gb_rmse, gb_r2 = evaluate_model(gb, X_test, y_test)

print("Gradient Boosting")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R2:", gb_r2)

Gradient Boosting
MAE: 0.8197410158494793
RMSE: 1.9502056091201898
R2: 0.7309315587101735


STEP 7 — Compare models

In [44]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "MAE": [lr_mae, rf_mae, gb_mae],
    "RMSE": [lr_rmse, rf_rmse, gb_rmse],
    "R2 Score": [lr_r2, rf_r2, gb_r2]
})

results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,0.452392,1.804445,0.769650
1,Random Forest,1.164160,2.214648,0.653015
2,Gradient Boosting,0.819741,1.950206,0.730932


STEP 8 — Select best model

In [45]:
best_model = rf

STEP 9 — Save best model

In [46]:
joblib.dump(best_model, "../model/student_performance_model.pkl")
print("Model saved successfully!")

Model saved successfully!
